
# GVH Diagonal Cubic `.28.21.2.1.2.1` — FAST
## Exact Cubic Algebraic Collision Rank Certificate

### Unique lock B1

The exact quintic/root problem is already closed.

The exact \(Q_2-Q_3\) crossing is already semisimple.

This notebook addresses **only** the two algebraic self-coalescences of the cubic factor \(F_3\).

The required exact statement at each collision is:

\[
\boxed{
\dim\ker\!\left(A_{\rm phys}^2-y_c I\right)=4.
}
\]

Equivalently:

\[
\boxed{
\operatorname{rank}\!\left(A_{\rm phys}^2-y_c I\right)=16.
}
\]

No SVD, floating tolerance, interpolation, or directional scan may decide this lock.

### Exact route used here

Instead of introducing nested radicals directly into the 20-dimensional physical frame, we work with the inherited 28-dimensional Hamilton-Dirac system and exact constraint matrix.

For an unnormalised edge direction \(d\), with \(r^2=d\cdot d\),

\[
\widehat A
=
\frac{1}{r}S A_{\rm HD}S^{-1},
\qquad
\widehat C=C_{\rm phys}S^{-1}.
\]

Thus:

\[
\ker(\widehat C)
\cap
\ker(\widehat A^2-y_cI)
\]

is isomorphic to:

\[
\ker(C_{\rm phys})
\cap
\ker(A_{\rm HD}^2-y_c r^2 I).
\]

Therefore define the exact stacked matrix

\[
\mathcal K_c=
\begin{pmatrix}
C_{\rm phys}\\
A_{\rm HD}^2-y_c r^2I
\end{pmatrix}.
\]

It has 28 columns, so:

\[
\boxed{
\operatorname{rank}\mathcal K_c=24
\iff
\dim\ker\mathcal K_c=4.
}
\]

This avoids all square-root normalisation artifacts.


In [1]:

from __future__ import annotations

import sys, json
from pathlib import Path

import sympy as sp
from sympy.polys.matrices import DomainMatrix

PARENT_2821212 = {
    "artifact":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2_"
        "Exact_Crossing_Semisimplicity_and_Uniform_Projector_Certificate_FAST_"
        "CORRECTED_COLAB(1).ipynb",
    "executed_size_bytes": 63741,
    "executed_sha256": 'eb1d43d6662476cb777b1b2f6efc7d090fc04d2b230bc39acd2064dcca0da8f9',
    "machine_clean": True,
    "q2_q3_crossing_semisimplicity_certified": True,
    "cubic_self_crossing_semisimplicity_certified": False,
    "global_crossing_semisimplicity_certified": False,
    "light_sector_global_semisimplicity_certified": False,
    "uniform_directional_projector_control_certified": False,
    "strong_hyperbolicity_proven": False,
}

G28212121_PROVENANCE_GATE_PASS = all([
    PARENT_2821212["machine_clean"],
    PARENT_2821212["q2_q3_crossing_semisimplicity_certified"],
    not PARENT_2821212["cubic_self_crossing_semisimplicity_certified"],
    not PARENT_2821212["strong_hyperbolicity_proven"],
])

assert G28212121_PROVENANCE_GATE_PASS

print("Python =",sys.version.split()[0])
print("SymPy =",sp.__version__)
print("G28212121_PROVENANCE_GATE_PASS =",G28212121_PROVENANCE_GATE_PASS)


Python = 3.13.15
SymPy = 1.14.0
G28212121_PROVENANCE_GATE_PASS = True



# 1. Inherited exact principal-symbol construction

The next five code cells are copied from the audited principal-symbol reconstruction used by the parent checkpoint.

They reconstruct:

- the raw principal pencil;
- the healthy anisotropic frozen witness;
- the 14-dimensional kinetic sector;
- the four diffeomorphism null directions;
- the global gauge matrix \(F(\mathbf n)\);
- the exact global gauge regularity determinant.

No new physical structure is introduced.


In [2]:

eta=sp.diag(-1,1,1,1)

a0,a1,a2=sp.symbols("a0 a1 a2",real=True)
a3=-a0-a1-a2

Abar=sp.diag(a0,a1,a2,a3)
Qbar=sp.factor(sp.trace(Abar*Abar))

KS,kappaD,Mpl2=sp.symbols(
    "K_S kappa_D Mpl2",
    real=True
)

names=[
    "n","beta1","beta2","beta3",
    "h11","h22","h33","h12","h13","h23",
    "D00","D01","D02","D03",
    "D11","D22","D33","D12","D13","D23",
]

h_basis=[]
D_basis=[]

for name in names:
    h=sp.zeros(4)
    d=sp.zeros(4)

    if name=="n":
        h[0,0]=-2
    elif name.startswith("beta"):
        i=int(name[-1])
        h[0,i]=h[i,0]=1
    elif name.startswith("h"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        h[i,j]=h[j,i]=1
    elif name.startswith("D"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        d[i,j]=d[j,i]=1

    h_basis.append(h)
    D_basis.append(d)

dA_basis=[]

for h,d in zip(h_basis,D_basis):
    dM=-eta*h*Abar+eta*d
    dA=dM-sp.trace(dM)*sp.eye(4)/4
    dA_basis.append(dA)

def build_principal_matrix(p):
    H=sp.zeros(20)

    # Pure-GVH-P sector
    for alpha_idx in range(4):
        B=[]
        J=[]
        C=[]

        for h,dA in zip(h_basis,dA_basis):
            Gamma=sp.zeros(4)

            for mu in range(4):
                for nu in range(4):
                    acc=0
                    for rho in range(4):
                        acc += eta[mu,rho]*(
                            p[alpha_idx]*h[rho,nu]
                            +p[nu]*h[rho,alpha_idx]
                            -p[rho]*h[alpha_idx,nu]
                        )/2
                    Gamma[mu,nu]=acc

            Bj=p[alpha_idx]*dA+Gamma*Abar-Abar*Gamma
            B.append(Bj)
            J.append(sp.trace(Abar*Bj))
            C.append(Abar*Bj-Bj*Abar)

        sign=eta[alpha_idx,alpha_idx]

        for j in range(20):
            for k in range(j,20):
                shape=Qbar*sp.trace(B[j]*B[k])-J[j]*J[k]
                angle=sp.trace(C[j]*C[k])

                val=(
                    -KS*sign*shape
                    -kappaD*sp.Rational(1,2)*sign*angle
                )

                H[j,k]+=val
                if k!=j:
                    H[k,j]+=val

    # Einstein-Hilbert / Fierz-Pauli benchmark
    pvec=sp.Matrix(p)
    pup=eta*pvec
    p2=(pvec.T*eta*pvec)[0]

    attrs=[]

    for h in h_basis[:10]:
        hup=eta*h*eta
        v=[
            sum(p[mu]*hup[mu,nu] for mu in range(4))
            for nu in range(4)
        ]
        w=[
            sum(pup[lam]*h[lam,nu] for lam in range(4))
            for nu in range(4)
        ]
        trh=sp.trace(eta*h)
        vp=sum(v[nu]*p[nu] for nu in range(4))
        attrs.append((h,hup,v,w,trh,vp))

    for j in range(10):
        hj,hjup,vj,wj,trj,vpj=attrs[j]

        for k in range(j,10):
            hk,hkup,vk,wk,trk,vpk=attrs[k]

            inner=sum(
                hj[mu,nu]*hkup[mu,nu]
                for mu in range(4)
                for nu in range(4)
            )

            BF=(
                p2*inner
                -sum(
                    vj[nu]*wk[nu]+vk[nu]*wj[nu]
                    for nu in range(4)
                )
                +vpj*trk
                +vpk*trj
                -p2*trj*trk
            )

            val=-Mpl2*sp.Rational(1,4)*BF

            H[j,k]+=val
            if k!=j:
                H[k,j]+=val

    return H

e0=(1,0,0,0)
e1=(0,1,0,0)
e2=(0,0,1,0)
e3=(0,0,0,1)

P_e0=build_principal_matrix(e0)
P_e1=build_principal_matrix(e1)
P_e2=build_principal_matrix(e2)
P_e3=build_principal_matrix(e3)

K_raw=P_e0

M_raw={
    1:build_principal_matrix((1,1,0,0))-P_e0-P_e1,
    2:build_principal_matrix((1,0,1,0))-P_e0-P_e2,
    3:build_principal_matrix((1,0,0,1))-P_e0-P_e3,
}

G_raw={
    (1,1):P_e1,
    (2,2):P_e2,
    (3,3):P_e3,
    (1,2):(build_principal_matrix((0,1,1,0))-P_e1-P_e2)/2,
    (1,3):(build_principal_matrix((0,1,0,1))-P_e1-P_e3)/2,
    (2,3):(build_principal_matrix((0,0,1,1))-P_e2-P_e3)/2,
}

G2821161_RAW_PENCIL_RECONSTRUCTED=all([
    K_raw.shape==(20,20),
    all(M_raw[i].shape==(20,20) for i in (1,2,3)),
    all(G_raw[key].shape==(20,20) for key in G_raw),
])

assert G2821161_RAW_PENCIL_RECONSTRUCTED

print("G2821161_RAW_PENCIL_RECONSTRUCTED =",G2821161_RAW_PENCIL_RECONSTRUCTED)


G2821161_RAW_PENCIL_RECONSTRUCTED = True


In [3]:
D_raw={i:sp.simplify(M_raw[i]/2) for i in (1,2,3)}
assert all(sp.simplify(D_raw[i]+D_raw[i].T-M_raw[i])==sp.zeros(20) for i in (1,2,3))
print("Principal Legendre representative D_i=M_i/2 fixed")

Principal Legendre representative D_i=M_i/2 fixed


In [4]:

healthy_subs={
    a0:sp.Rational(3,4),
    a1:-sp.Rational(1,5),
    a2:-sp.Rational(1,4),
    KS:1,
    kappaD:2,
    Mpl2:1,
}

K_h=K_raw.subs(healthy_subs)
M_h={i:M_raw[i].subs(healthy_subs) for i in (1,2,3)}
D_h={i:D_raw[i].subs(healthy_subs) for i in (1,2,3)}
G_h={key:G_raw[key].subs(healthy_subs) for key in G_raw}

R_kin=sp.Matrix.hstack(*K_h.columnspace())
K14=sp.simplify(R_kin.T*K_h*R_kin)
K14_inv=K14.inv()

a=[a0,a1,a2,a3]

def original_gauge_vectors(p):
    vectors=[]

    for sigma in range(4):
        zeta=[0,0,0,0]
        zeta[sigma]=1

        hg=sp.zeros(4)
        dD=sp.zeros(4)

        for mu in range(4):
            for nu in range(4):
                hg[mu,nu]=(
                    p[mu]*zeta[nu]
                    +p[nu]*zeta[mu]
                )

                dD[mu,nu]=(
                    a[nu]*p[mu]*zeta[nu]
                    +a[mu]*p[nu]*zeta[mu]
                )

        vec=sp.zeros(20,1)

        vec[0]=-hg[0,0]/2
        vec[1]=hg[0,1]
        vec[2]=hg[0,2]
        vec[3]=hg[0,3]
        vec[4]=hg[1,1]
        vec[5]=hg[2,2]
        vec[6]=hg[3,3]
        vec[7]=hg[1,2]
        vec[8]=hg[1,3]
        vec[9]=hg[2,3]

        vals=[
            dD[0,0],dD[0,1],dD[0,2],dD[0,3],
            dD[1,1],dD[2,2],dD[3,3],
            dD[1,2],dD[1,3],dD[2,3],
        ]

        for j,val in enumerate(vals,start=10):
            vec[j]=val

        vectors.append(vec)

    trace=sp.zeros(20,1)
    trace[10]=-1
    trace[14]=1
    trace[15]=1
    trace[16]=1

    vectors.append(trace)

    return vectors

N_diff=sp.Matrix.hstack(
    *original_gauge_vectors((1,0,0,0))[:4]
).subs(healthy_subs)

N_trace=sp.zeros(20,1)
N_trace[10]=-1
N_trace[14]=1
N_trace[15]=1
N_trace[16]=1

N_radial=sp.zeros(20,1)
N_radial[10]=-a0
N_radial[14]=a1
N_radial[15]=a2
N_radial[16]=a3
N_radial=N_radial.subs(healthy_subs)

N6=sp.Matrix.hstack(
    N_diff,
    N_trace,
    N_radial,
)

T20=sp.Matrix.hstack(
    R_kin,
    N6,
)

G2821161_ORIGINAL_NULL_BASIS_PASS=all([
    K_h.rank()==14,
    R_kin.shape==(20,14),
    R_kin.rank()==14,
    N6.shape==(20,6),
    N6.rank()==6,
    K_h*N6==sp.zeros(20,6),
    T20.rank()==20,
])

assert G2821161_ORIGINAL_NULL_BASIS_PASS

print("rank K_h =",K_h.rank())
print("rank N6 =",N6.rank())
print("rank T20 =",T20.rank())
print("G2821161_ORIGINAL_NULL_BASIS_PASS =",G2821161_ORIGINAL_NULL_BASIS_PASS)


rank K_h = 14
rank N6 = 6
rank T20 = 20
G2821161_ORIGINAL_NULL_BASIS_PASS = True


In [5]:
n1,n2,n3=sp.symbols("n1 n2 n3",real=True)

B_symbolic=(
    n1*M_h[1]
    +n2*M_h[2]
    +n3*M_h[3]
)

C_symbolic=(
    n1**2*G_h[(1,1)]
    +n2**2*G_h[(2,2)]
    +n3**2*G_h[(3,3)]
    +2*n1*n2*G_h[(1,2)]
    +2*n1*n3*G_h[(1,3)]
    +2*n2*n3*G_h[(2,3)]
)

G0_symbolic=sp.Matrix.hstack(
    *original_gauge_vectors((0,n1,n2,n3))[:4]
).subs(healthy_subs)

G1_symbolic=N_diff

noether_checks=[
    sp.simplify(K_h*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(K_h*G0_symbolic+B_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(B_symbolic*G0_symbolic+C_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(C_symbolic*G0_symbolic)==sp.zeros(20,4),
]

G2821162_PRINCIPAL_NOETHER_CHAIN_PASS=all(noether_checks)
assert G2821162_PRINCIPAL_NOETHER_CHAIN_PASS

T20_inv=T20.inv()
K14_inv=K14.inv()

print("Noether checks =",noether_checks)

Noether checks = [True, True, True, True]


In [6]:
Q_symbolic=sp.simplify(
    (T20_inv*G0_symbolic)[:14,:]
)
F_global_symbolic=sp.simplify(Q_symbolic.T)
Gram_global=sp.simplify(Q_symbolic.T*Q_symbolic)
det_Gram=sp.factor(Gram_global.det())

poly_Gram=sp.Poly(sp.expand(det_Gram),n1,n2,n3)
gram_terms=poly_Gram.terms()

gram_even_exponents=all(
    all(e%2==0 for e in monom)
    for monom,coeff in gram_terms
)
gram_positive_coeffs=all(
    bool(coeff>0)
    for monom,coeff in gram_terms
)

G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS=all([
    Q_symbolic.shape==(14,4),
    gram_even_exponents,
    gram_positive_coeffs,
    len(gram_terms)>0,
])

assert G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS

print("det Gram total degree =",poly_Gram.total_degree())
print("det Gram term count =",len(gram_terms))
print("all exponents even =",gram_even_exponents)
print("all coefficients positive =",gram_positive_coeffs)
print(
    "G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS =",
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS
)

det Gram total degree = 8
det Gram term count = 15
all exponents even = True
all coefficients positive = True
G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS = True



# 2. Exact cubic collision data

The exact residual factorisation is inherited as:

\[
Q_5\propto F_2F_3.
\]

The two cubic self-coalescences are:

### Edge \(v=0\)

\[
u=u_c^{(v=0)}
=
-\frac{284832107}{268295000}
+
\frac{395941}{36488120000}\sqrt{16205811493},
\]

with double squared-speed:

\[
y_c^{(v=0)}
=
\frac{1035103}{1395134}
+
\frac{133}{23717278}\sqrt{16205811493}.
\]

### Edge \(u=0\)

\[
v=v_c^{(u=0)}
=
-\frac{150509593748759}{35641420508441}
+
\frac{12815600}{35641420508441}
\sqrt{161292621824569},
\]

with:

\[
y_c^{(u=0)}
=
\frac{4021300195103}{42121678782703}
+
\frac{5625200}{42121678782703}
\sqrt{161292621824569}.
\]

We first verify exactly that each is a **double but not triple** root of \(F_3\), and that \(F_2\) is nonzero there.


In [7]:

u,v,y=sp.symbols("u v y",real=True)

F2=sp.expand(sp.sympify(
    '3575880000*u*v - 85956352000000*u*y + 85328549440000*u + 1975584303*v**2 - 46004626366567*v*y + 45235610405161*v + 98557536706400*y**2 - 294159043526233*y + 213754531196936',
    locals={"u":u,"v":v,"y":y}
))
F3=sp.expand(sp.sympify(
    '475200000000*u**2*v + 65062400000000*u**2*y - 86129600000000*u**2 + 514487160000*u*v**2 + 74296351360000*u*v*y - 90887492080000*u*v - 204837783800000*u*y**2 + 588610947160000*u*y - 428268669800000*u + 139196650824*v**3 + 21092071871849*v**2*y - 23950046355521*v**2 - 114531345426641*v*y**2 + 327629553487984*v*y - 228922045526471*v + 152750499165134*y**3 - 699211437911961*y**2 + 1060470112941969*y - 532367422897166',
    locals={"u":u,"v":v,"y":y}
))

D_v0=sp.Integer(16205811493)
D_u0=sp.Integer(161292621824569)

u_c_v0=(
    -sp.Rational(284832107,268295000)
    +sp.Rational(395941,36488120000)*sp.sqrt(D_v0)
)
y_c_v0=(
    sp.Rational(1035103,1395134)
    +sp.Rational(133,23717278)*sp.sqrt(D_v0)
)

v_c_u0=(
    -sp.Rational(150509593748759,35641420508441)
    +sp.Rational(12815600,35641420508441)*sp.sqrt(D_u0)
)
y_c_u0=(
    sp.Rational(4021300195103,42121678782703)
    +sp.Rational(5625200,42121678782703)*sp.sqrt(D_u0)
)

def exact_double_root_gate(subs):
    return {
        "F3_zero":
            sp.simplify(F3.subs(subs))==0,
        "dF3_zero":
            sp.simplify(sp.diff(F3,y).subs(subs))==0,
        "ddF3_nonzero":
            sp.simplify(sp.diff(F3,y,2).subs(subs))!=0,
        "F2_nonzero":
            sp.simplify(F2.subs(subs))!=0,
    }

gate_v0=exact_double_root_gate({u:u_c_v0,v:0,y:y_c_v0})
gate_u0=exact_double_root_gate({u:0,v:v_c_u0,y:y_c_u0})

G28212121_EXACT_CUBIC_DOUBLE_ROOT_LOCI_PASS=all(
    list(gate_v0.values())+list(gate_u0.values())
)

assert G28212121_EXACT_CUBIC_DOUBLE_ROOT_LOCI_PASS

print("v=0 gate =",gate_v0)
print("u=0 gate =",gate_u0)
print(
    "G28212121_EXACT_CUBIC_DOUBLE_ROOT_LOCI_PASS =",
    G28212121_EXACT_CUBIC_DOUBLE_ROOT_LOCI_PASS
)


v=0 gate = {'F3_zero': True, 'dF3_zero': True, 'ddF3_nonzero': True, 'F2_nonzero': True}
u=0 gate = {'F3_zero': True, 'dF3_zero': True, 'ddF3_nonzero': True, 'F2_nonzero': True}
G28212121_EXACT_CUBIC_DOUBLE_ROOT_LOCI_PASS = True



# 3. Quartic number fields for the edge direction ratios

Use unnormalised directions:

\[
d_{v=0}=(q,0,1),
\qquad
d_{u=0}=(0,q,1).
\]

Then:

\[
u=\frac{q^2}{1+q^2}
\quad\text{or}\quad
v=\frac{q^2}{1+q^2}.
\]

At the two collision points, \(q\) satisfies an irreducible quartic over \(\mathbb Q\).

For \(v=0\):

\[
\boxed{
3418395259050057q^4
+
818767593940114q^2
-
1140102865109943
=0.
}
\]

For \(u=0\):

\[
\boxed{
2518900800000000q^4
+
942479286341600q^2
-
1184365888065549
=0.
}
\]

The physical positive square roots are embeddings of these exact quartic number fields.

Rank over the abstract field is invariant under field embedding, so we do not need to choose a floating approximation to the physical root.


In [8]:

q=sp.symbols("q")

q_phys_v0=sp.sqrt(
    sp.simplify(u_c_v0/(1-u_c_v0))
)
q_phys_u0=sp.sqrt(
    sp.simplify(v_c_u0/(1-v_c_u0))
)

p_v0=sp.Poly(
    3418395259050057*q**4
    +818767593940114*q**2
    -1140102865109943,
    q,
    domain=sp.QQ,
)

p_u0=sp.Poly(
    2518900800000000*q**4
    +942479286341600*q**2
    -1184365888065549,
    q,
    domain=sp.QQ,
)

minpoly_v0=sp.Poly(
    sp.minpoly(q_phys_v0,q),
    q,
    domain=sp.QQ,
)
minpoly_u0=sp.Poly(
    sp.minpoly(q_phys_u0,q),
    q,
    domain=sp.QQ,
)

G28212121_QUARTIC_FIELD_V0_PASS=all([
    p_v0.is_irreducible,
    p_v0.monic()==minpoly_v0.monic(),
])

G28212121_QUARTIC_FIELD_U0_PASS=all([
    p_u0.is_irreducible,
    p_u0.monic()==minpoly_u0.monic(),
])

assert G28212121_QUARTIC_FIELD_V0_PASS
assert G28212121_QUARTIC_FIELD_U0_PASS

print("p_v0 irreducible =",p_v0.is_irreducible)
print("p_u0 irreducible =",p_u0.is_irreducible)
print(
    "G28212121_QUARTIC_FIELD_V0_PASS =",
    G28212121_QUARTIC_FIELD_V0_PASS
)
print(
    "G28212121_QUARTIC_FIELD_U0_PASS =",
    G28212121_QUARTIC_FIELD_U0_PASS
)


p_v0 irreducible = True
p_u0 irreducible = True
G28212121_QUARTIC_FIELD_V0_PASS = True
G28212121_QUARTIC_FIELD_U0_PASS = True



# 4. Exact \(y_c\) relations inside the quartic fields

The collision squared-speeds belong to the same quadratic subfields.

Exactly:

\[
\boxed{
y_c^{(v=0)}
=
\frac{99893}{77402}
+
\frac{20000}{38701}u_c
}
\]

and:

\[
\boxed{
y_c^{(u=0)}
=
\frac{99002}{59501}
+
\frac{22099}{59501}v_c.
}
\]

Therefore inside the quartic field we may write \(y_c\) using only \(q^2/(1+q^2)\), with no separate radical embedding.


In [9]:

y_v0_a=sp.Rational(99893,77402)
y_v0_b=sp.Rational(20000,38701)

y_u0_a=sp.Rational(99002,59501)
y_u0_b=sp.Rational(22099,59501)

G28212121_Y_RELATIONS_PASS=all([
    sp.simplify(
        y_c_v0-(y_v0_a+y_v0_b*u_c_v0)
    )==0,
    sp.simplify(
        y_c_u0-(y_u0_a+y_u0_b*v_c_u0)
    )==0,
])

assert G28212121_Y_RELATIONS_PASS

print(
    "G28212121_Y_RELATIONS_PASS =",
    G28212121_Y_RELATIONS_PASS
)


G28212121_Y_RELATIONS_PASS = True



# 5. Exact algebraic-field Hamilton-Dirac certificate

All matrix arithmetic below is performed over:

\[
\mathbb Q[q]/(p(q)).
\]

The algorithm:

1. converts the fixed rational GVH matrices into the algebraic field;
2. builds \(B,D,C,F\) exactly at \(d=(q,0,1)\) or \(d=(0,q,1)\);
3. solves the multiplier blocks exactly;
4. constructs \(A_{\rm HD}\) and \(C_{\rm phys}\);
5. checks:
   \[
   \operatorname{rank}C_{\rm phys}=8;
   \]
6. checks exact invariance of the physical constraint kernel:
   \[
   \operatorname{rank}
   \begin{pmatrix}
   C_{\rm phys}\\
   C_{\rm phys}A_{\rm HD}
   \end{pmatrix}
   =8;
   \]
7. computes:
   \[
   \operatorname{rank}
   \begin{pmatrix}
   C_{\rm phys}\\
   A_{\rm HD}^2-y_cr^2I
   \end{pmatrix}.
   \]

There are no floating-point operations in the certificate.


In [10]:

def _dm(M,K):
    return DomainMatrix.from_Matrix(M).convert_to(K)


def exact_edge_collision_certificate(
    minimal_poly,
    edge,
    y_a,
    y_b,
):
    K=sp.QQ.alg_field_from_poly(minimal_poly.monic())
    alpha=K.convert(K.ext)

    one=K.one
    two=K.convert(2)
    minus_one=K.convert(-1)
    half=K.convert(sp.Rational(1,2))

    Mh={
        i:_dm(M_h[i],K)
        for i in (1,2,3)
    }
    Gh={
        key:_dm(G_h[key],K)
        for key in G_h
    }

    R=_dm(R_kin,K)
    N=_dm(N_diff,K)
    Kinv=_dm(K14_inv,K)

    Fcoef={
        1:_dm(
            F_global_symbolic.subs({
                n1:1,n2:0,n3:0
            }),
            K,
        ),
        2:_dm(
            F_global_symbolic.subs({
                n1:0,n2:1,n3:0
            }),
            K,
        ),
        3:_dm(
            F_global_symbolic.subs({
                n1:0,n2:0,n3:1
            }),
            K,
        ),
    }

    if edge=="xz":
        B=Mh[1]*alpha+Mh[3]
        C=(
            Gh[(1,1)]*(alpha*alpha)
            +Gh[(3,3)]
            +Gh[(1,3)]*(two*alpha)
        )
        F=Fcoef[1]*alpha+Fcoef[3]

    elif edge=="yz":
        B=Mh[2]*alpha+Mh[3]
        C=(
            Gh[(2,2)]*(alpha*alpha)
            +Gh[(3,3)]
            +Gh[(2,3)]*(two*alpha)
        )
        F=Fcoef[2]*alpha+Fcoef[3]

    else:
        raise ValueError("edge must be 'xz' or 'yz'")

    D=B*half

    Rt=R.transpose()
    Nt=N.transpose()

    Baa=Rt*B*R
    BaN=Rt*B*N

    Daa=Rt*D*R
    Dau=Rt*D*N

    Caa=Rt*C*R
    CaN=Rt*C*N

    BNR=Nt*B*R
    CNR=Nt*C*R
    CNN=Nt*C*N

    Umat=F*Kinv*Dau
    Uinv=Umat.inv()

    Ux=(Uinv*F*Kinv*Daa)*minus_one
    Up=Uinv*F*Kinv

    Vx=Kinv*((Daa+Dau*Ux)*minus_one)
    Vp=Kinv*(
        DomainMatrix.eye(14,K)-Dau*Up
    )

    Lmat=F*Kinv*BaN
    Linv=Lmat.inv()

    rhs_x=F*Kinv*(
        Baa*Vx+Caa+CaN*Ux
    )
    rhs_p=F*Kinv*(
        Baa*Vp+CaN*Up
    )

    Lx=(Linv*rhs_x)*minus_one
    Lp=(Linv*rhs_p)*minus_one

    Pdx=(
        Daa*Vx
        +Dau*Lx
        +Caa
        +CaN*Ux
    )*minus_one

    Pdp=(
        Daa*Vp
        +Dau*Lp
        +CaN*Up
    )*minus_one

    AHD=DomainMatrix.vstack(
        DomainMatrix.hstack(Vx,Vp),
        DomainMatrix.hstack(Pdx,Pdp),
    )

    Hx=BNR*Vx+CNR+CNN*Ux
    Hp=BNR*Vp+CNN*Up

    Z=DomainMatrix.zeros((4,14),K)

    Cphys=DomainMatrix.vstack(
        DomainMatrix.hstack(F,Z),
        DomainMatrix.hstack(Hx,Hp),
    )

    q2=alpha*alpha
    coord=q2/(one+q2)

    yc=(
        K.convert(y_a)
        +K.convert(y_b)*coord
    )

    r2=one+q2
    target=yc*r2

    collision_matrix=(
        AHD*AHD
        -DomainMatrix.eye(28,K)*target
    )

    invariance_stack=DomainMatrix.vstack(
        Cphys,
        Cphys*AHD,
    )

    kernel_stack=DomainMatrix.vstack(
        Cphys,
        collision_matrix,
    )

    constraint_rank=Cphys.rank()
    invariance_rank=invariance_stack.rank()
    collision_stack_rank=kernel_stack.rank()
    collision_kernel_dim=28-collision_stack_rank

    return {
        "field":str(K),
        "minimal_poly":str(minimal_poly.as_expr()),
        "constraint_rank":int(constraint_rank),
        "constraint_kernel_invariant":
            bool(invariance_rank==constraint_rank),
        "invariance_stack_rank":
            int(invariance_rank),
        "collision_stack_rank":
            int(collision_stack_rank),
        "collision_kernel_dim":
            int(collision_kernel_dim),
        "U_invertible":bool(Umat.det()!=K.zero),
        "Lambda_invertible":bool(Lmat.det()!=K.zero),
    }



# 6. Exact certificate — cubic collision on \(v=0\)

Required:

\[
\boxed{
\operatorname{rank}\mathcal K_c=24
}
\]

and therefore:

\[
\boxed{
\dim\ker\mathcal K_c=4.
}
\]


In [11]:

cert_v0=exact_edge_collision_certificate(
    p_v0,
    "xz",
    y_v0_a,
    y_v0_b,
)

G28212121_CUBIC_V0_COLLISION_RANK_CERTIFIED=all([
    cert_v0["constraint_rank"]==8,
    cert_v0["constraint_kernel_invariant"],
    cert_v0["U_invertible"],
    cert_v0["Lambda_invertible"],
    cert_v0["collision_stack_rank"]==24,
    cert_v0["collision_kernel_dim"]==4,
])

assert G28212121_CUBIC_V0_COLLISION_RANK_CERTIFIED

print(json.dumps(cert_v0,indent=2))
print(
    "G28212121_CUBIC_V0_COLLISION_RANK_CERTIFIED =",
    G28212121_CUBIC_V0_COLLISION_RANK_CERTIFIED
)


{
  "field": "QQ<CRootOf(3418395259050057*q**4 + 818767593940114*q**2 - 1140102865109943, 3)>",
  "minimal_poly": "3418395259050057*q**4 + 818767593940114*q**2 - 1140102865109943",
  "constraint_rank": 8,
  "constraint_kernel_invariant": true,
  "invariance_stack_rank": 8,
  "collision_stack_rank": 24,
  "collision_kernel_dim": 4,
  "U_invertible": true,
  "Lambda_invertible": true
}
G28212121_CUBIC_V0_COLLISION_RANK_CERTIFIED = True



# 7. Exact certificate — cubic collision on \(u=0\)

The second algebraic collision is treated independently in its own quartic number field.

Again the required exact result is:

\[
\boxed{
\operatorname{rank}\mathcal K_c=24,
\qquad
\dim\ker\mathcal K_c=4.
}
\]


In [12]:

cert_u0=exact_edge_collision_certificate(
    p_u0,
    "yz",
    y_u0_a,
    y_u0_b,
)

G28212121_CUBIC_U0_COLLISION_RANK_CERTIFIED=all([
    cert_u0["constraint_rank"]==8,
    cert_u0["constraint_kernel_invariant"],
    cert_u0["U_invertible"],
    cert_u0["Lambda_invertible"],
    cert_u0["collision_stack_rank"]==24,
    cert_u0["collision_kernel_dim"]==4,
])

assert G28212121_CUBIC_U0_COLLISION_RANK_CERTIFIED

print(json.dumps(cert_u0,indent=2))
print(
    "G28212121_CUBIC_U0_COLLISION_RANK_CERTIFIED =",
    G28212121_CUBIC_U0_COLLISION_RANK_CERTIFIED
)


{
  "field": "QQ<CRootOf(2518900800000000*q**4 + 942479286341600*q**2 - 1184365888065549, 3)>",
  "minimal_poly": "2518900800000000*q**4 + 942479286341600*q**2 - 1184365888065549",
  "constraint_rank": 8,
  "constraint_kernel_invariant": true,
  "invariance_stack_rank": 8,
  "collision_stack_rank": 24,
  "collision_kernel_dim": 4,
  "U_invertible": true,
  "Lambda_invertible": true
}
G28212121_CUBIC_U0_COLLISION_RANK_CERTIFIED = True



# 8. Semisimplicity conclusion

At each cubic collision, \(F_3\) has a double root \(y_c>0\), not a triple root, and \(F_2(y_c)\neq0\).

Thus on the physical 20-dimensional first-order symbol:

\[
\lambda=\pm\sqrt{y_c}
\]

each has algebraic multiplicity \(2\).

The exact result:

\[
\dim\ker(A_{\rm phys}^2-y_cI)=4
\]

is the sum of the two eigenspace dimensions.

Since each individual eigenspace has dimension at most its algebraic multiplicity \(2\), the sum can equal \(4\) only if:

\[
\boxed{
\dim E_{+\sqrt{y_c}}=2,
\qquad
\dim E_{-\sqrt{y_c}}=2.
}
\]

Therefore both cubic self-coalescences are semisimple.

Combined with the inherited exact \(Q_2-Q_3\) crossing certificate, the residual crossing/coalescence semisimplicity lock is closed.


In [13]:

G28212121_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED=all([
    G28212121_CUBIC_V0_COLLISION_RANK_CERTIFIED,
    G28212121_CUBIC_U0_COLLISION_RANK_CERTIFIED,
])

G28212121_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED=all([
    PARENT_2821212[
        "q2_q3_crossing_semisimplicity_certified"
    ],
    G28212121_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED,
])

assert G28212121_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED
assert G28212121_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED

print(
    "G28212121_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    G28212121_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED
)
print(
    "G28212121_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    G28212121_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED
)


G28212121_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED = True
G28212121_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED = True



# 9. What remains open

B1 is now closed.

The following remain independent blockers:

## B2 — light-sector global semisimplicity

Prove direction-globally:

\[
\boxed{
\operatorname{rank}(A_{\rm phys}-I)=15,
\qquad
\operatorname{rank}(A_{\rm phys}+I)=15.
}
\]

A finite exact atlas of nonvanishing \(15\times15\) minors is an authorised route.

## B3 — uniform projector/diagonalizer control

Prove a direction-uniform bound, including all collision neighbourhoods:

\[
\boxed{
\sup_{\mathbf n}
\|S(\mathbf n)\|
\|S(\mathbf n)^{-1}\|<\infty
}
\]

or an equivalent exact projector certificate.

Therefore:

\[
\boxed{
\texttt{STRONG\_HYPERBOLICITY\_PROVEN=False}
}
\]

must remain unchanged.


In [14]:

G28212121_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED=False
G28212121_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED=False

G28212121_STRONG_HYPERBOLICITY_PROVEN=all([
    G28212121_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED,
    G28212121_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED,
    G28212121_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED,
])

assert not G28212121_STRONG_HYPERBOLICITY_PROVEN

G28212121_NEXT_AUTHORIZED=(
    ".28.21.2.1.2.2 — exact global light-sector rank-15 atlas"
)

print(
    "G28212121_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED =",
    G28212121_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED
)
print(
    "G28212121_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED =",
    G28212121_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED
)
print(
    "G28212121_STRONG_HYPERBOLICITY_PROVEN =",
    G28212121_STRONG_HYPERBOLICITY_PROVEN
)
print("NEXT_AUTHORIZED =",G28212121_NEXT_AUTHORIZED)


G28212121_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED = False
G28212121_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED = False
G28212121_STRONG_HYPERBOLICITY_PROVEN = False
NEXT_AUTHORIZED = .28.21.2.1.2.2 — exact global light-sector rank-15 atlas



# 10. Four-level protocol

## Level 1 — GVH

Only the already-derived pure-GVH-P physical principal system is used.

No new action, coupling, metric, or dynamics is introduced.

## Level 2 — exact mathematics

The certificate uses exact number fields and exact matrix ranks.

No numerical tolerance decides semisimplicity.

## Level 3 — diagnostics

No scan is used.

No SVD or floating rank is used.

## Level 4 — units / observables

No SI scale, phenomenology, or observable is introduced.

This notebook closes only the algebraic collision-rank lock.


In [15]:

ESTABLISHED_PHYSICS_USED_AS_BENCHMARK_NOT_SUBSTITUTE=True

LEVEL1_GVH_PASS=True
LEVEL2_EXACT_ALGEBRA_PASS=True
LEVEL3_NO_NUMERICAL_RANK_GATE_PASS=True

UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK=0
NUMERICAL_SI_CALIBRATION_AUTHORIZED=False
LEVEL4_SI_LEDGER_PASS=True

FOUR_LEVEL_PROTOCOL_PASS=all([
    LEVEL1_GVH_PASS,
    LEVEL2_EXACT_ALGEBRA_PASS,
    LEVEL3_NO_NUMERICAL_RANK_GATE_PASS,
    LEVEL4_SI_LEDGER_PASS,
])

assert FOUR_LEVEL_PROTOCOL_PASS

print("FOUR_LEVEL_PROTOCOL_PASS =",FOUR_LEVEL_PROTOCOL_PASS)


FOUR_LEVEL_PROTOCOL_PASS = True


In [16]:

verdict={
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.1_"
        "Exact_Cubic_Algebraic_Collision_Rank_Certificate_FAST",
    "parent_28_21_2_1_2":PARENT_2821212,
    "scope":{
        "background":
            "fixed healthy local frozen spectral-diagonal anisotropic witness",
        "direction_domain":
            "two exact cubic discriminant-zero edge loci",
        "global_parameter_space_claim":False,
    },
    "exact":{
        "double_root_loci_pass":
            bool(G28212121_EXACT_CUBIC_DOUBLE_ROOT_LOCI_PASS),
        "quartic_field_v0_pass":
            bool(G28212121_QUARTIC_FIELD_V0_PASS),
        "quartic_field_u0_pass":
            bool(G28212121_QUARTIC_FIELD_U0_PASS),
        "y_relations_pass":
            bool(G28212121_Y_RELATIONS_PASS),
        "v0_collision":cert_v0,
        "u0_collision":cert_u0,
        "cubic_v0_collision_rank_certified":
            bool(G28212121_CUBIC_V0_COLLISION_RANK_CERTIFIED),
        "cubic_u0_collision_rank_certified":
            bool(G28212121_CUBIC_U0_COLLISION_RANK_CERTIFIED),
        "cubic_self_crossing_semisimplicity_certified":
            bool(G28212121_CUBIC_SELF_CROSSING_SEMISIMPLICITY_CERTIFIED),
        "global_crossing_semisimplicity_certified":
            bool(G28212121_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED),
    },
    "locks":{
        "light_sector_global_semisimplicity_certified":
            bool(G28212121_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED),
        "uniform_directional_projector_control_certified":
            bool(G28212121_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED),
        "strong_hyperbolicity_proven":
            bool(G28212121_STRONG_HYPERBOLICITY_PROVEN),
    },
    "protocol":{
        "four_level_protocol_pass":
            bool(FOUR_LEVEL_PROTOCOL_PASS),
        "universal_theory_selected_SI_scale_rank":0,
    },
    "status":
        "PASS_EXACT_CUBIC_ALGEBRAIC_COLLISION_RANKS_"
        "GLOBAL_RESIDUAL_CROSSING_SEMISIMPLICITY_CLOSED_"
        "LIGHT_PROJECTOR_OPEN",
    "next_authorized":G28212121_NEXT_AUTHORIZED,
}

export_dir=Path("/mnt/data/gvh_exports_28212121")
export_dir.mkdir(parents=True,exist_ok=True)

verdict_path=export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.1_"
    "Exact_Cubic_Algebraic_Collision_Rank_Certificate_FAST.json"
)

verdict_path.write_text(
    json.dumps(
        verdict,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("STATUS =",verdict["status"])
print(
    "CUBIC_V0_COLLISION_RANK_CERTIFIED =",
    verdict["exact"]["cubic_v0_collision_rank_certified"]
)
print(
    "CUBIC_U0_COLLISION_RANK_CERTIFIED =",
    verdict["exact"]["cubic_u0_collision_rank_certified"]
)
print(
    "GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    verdict["exact"]["global_crossing_semisimplicity_certified"]
)
print(
    "STRONG_HYPERBOLICITY_PROVEN =",
    verdict["locks"]["strong_hyperbolicity_proven"]
)
print("verdict JSON =",verdict_path)


STATUS = PASS_EXACT_CUBIC_ALGEBRAIC_COLLISION_RANKS_GLOBAL_RESIDUAL_CROSSING_SEMISIMPLICITY_CLOSED_LIGHT_PROJECTOR_OPEN
CUBIC_V0_COLLISION_RANK_CERTIFIED = True
CUBIC_U0_COLLISION_RANK_CERTIFIED = True
GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED = True
STRONG_HYPERBOLICITY_PROVEN = False
verdict JSON = /mnt/data/gvh_exports_28212121/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.1_Exact_Cubic_Algebraic_Collision_Rank_Certificate_FAST.json
